# QUN Sticker Studio — Kaggle edition

Первый бесплатный эксперимент с авторским набором **Omnii — добро рядом**.

## Как пользоваться
1. В Kaggle создай Notebook и импортируй этот файл .ipynb из GitHub.
2. В настройках Notebook включи **Accelerator → GPU** и **Internet → On**. Если доступен выбор GPU, начни с P100 или T4.
3. Если модель Stable Diffusion запрашивает доступ, прими условия в Hugging Face и добавь токен в **Kaggle Secrets** под именем **HF_TOKEN**.
4. Запусти ячейки сверху вниз. Сначала создаются четыре эталонных кандидата, затем три тестовых стикера.
5. Проверь тесты. Только если образ Omnii стабилен, измени значение **RUN_FULL_PACK = False** на **True** в ячейке полного набора и выполни её повторно.
6. Последние ячейки удалят фон, добавят единый контур, соберут превью и ZIP.

## Важные ограничения
- Этот ноутбук использует **Diffusers + Stable Diffusion 1.5 + IP-Adapter**, а не запускает визуальный интерфейс ComfyUI. Так мы начинаем без установки Python 3.12 на домашний компьютер и без сетевого туннеля.
- Модель может менять детали персонажа. IP-Adapter помогает сохранять сходство, но не гарантирует идеальную идентичность; проверь все 24 изображения.
- Фон удаляется автоматическим инструментом, края нужно осмотреть. Никакая генерация не публикуется автоматически.
- Бесплатный GPU в Kaggle зависит от квоты и доступности. Сохраняй результаты в /kaggle/working и скачай ZIP до окончания сессии.
- Перед коммерческим использованием проверь лицензии модели и IP-Adapter, а также права на конечные изображения.

## Спецификация QUN
- Правила: https://github.com/Serega40in/QUN/blob/main/docs/stickers/README.md
- 24 подробных промта: https://github.com/Serega40in/QUN/blob/main/docs/stickers/PROMPTS_OMNII_01.md
- Мастер-инструкция: https://github.com/Serega40in/QUN/blob/main/docs/stickers/MASTER_GENERATION_BRIEF.md


In [ ]:
# Install notebook dependencies. Kaggle supplies the Python runtime and CUDA-enabled PyTorch.
%pip install -q "diffusers>=0.32.2,<1.0.0" "transformers>=4.44,<5.0" accelerate safetensors scipy pillow rembg


In [ ]:
from pathlib import Path
import os, json, time, random, shutil, zipfile, math
import torch
from PIL import Image, ImageDraw, ImageFont, ImageFilter, ImageChops
from IPython.display import display, Markdown

WORK = Path("/kaggle/working/qun_omnii_01")
REF_DIR = WORK / "references"
TEST_DIR = WORK / "test"
SOURCE_DIR = WORK / "source"
FINAL_DIR = WORK / "telegram"
for folder in (REF_DIR, TEST_DIR, SOURCE_DIR, FINAL_DIR):
    folder.mkdir(parents=True, exist_ok=True)

print("GPU available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Torch:", torch.__version__)
else:
    raise RuntimeError("GPU не подключён. Открой Settings → Accelerator → GPU и перезапусти сессию.")

# Optional Hugging Face token stored securely in Kaggle Secrets as HF_TOKEN.
HF_TOKEN = os.environ.get("HF_TOKEN")
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN") or HF_TOKEN
except Exception:
    pass

MODEL_ID = "stable-diffusion-v1-5/stable-diffusion-v1-5"
ADAPTER_REPO = "h94/IP-Adapter"
ADAPTER_FILE = "ip-adapter_sd15.bin"
BASE_SEED = 4061109

print("Workspace:", WORK)
print("Model:", MODEL_ID)
print("HF token found:", bool(HF_TOKEN))


In [ ]:
from diffusers import StableDiffusionPipeline

# Keep the default safety checker enabled. P100/T4 supports fp16.
try:
    pipe = StableDiffusionPipeline.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float16,
        use_safetensors=True,
        token=HF_TOKEN or None,
        cache_dir="/kaggle/working/hf_cache",
    )
except Exception as exc:
    print("Не удалось загрузить базовую модель:", repr(exc))
    print("Если ошибка говорит о gated/restricted model: открой карточку модели на Hugging Face, прими условия, создай Read token и добавь его в Kaggle Secrets как HF_TOKEN.")
    print("Model card: https://huggingface.co/" + MODEL_ID)
    raise

pipe = pipe.to("cuda")
pipe.enable_attention_slicing()
pipe.enable_vae_slicing()
pipe.enable_vae_tiling()
print("Pipeline loaded. Safety checker enabled:", pipe.safety_checker is not None)


In [ ]:
CHARACTER_LOCK = (
    "The exact same original mascot character in every image: a tiny adorable fluffy round creature named Omnii, "
    "soft ivory-white plush fur with a slightly shaggy silhouette, very large expressive pale ice-blue faceted crystal eyes "
    "with gentle highlights, tiny simple mouth, two tiny rounded arms and short feet, friendly warm personality, "
    "consistent head-to-body proportions, same eye shape and same fur texture in every image. "
    "Premium polished 3D animated collectible character, soft tactile materials, clean smooth lighting, "
    "restrained QUN palette accents of deep blue, emerald green and soft gold only when needed."
)
COMPOSITION_LOCK = (
    "One single full-body character, centered, large and readable at small chat size, clear silhouette, "
    "entire character and props fully inside a square canvas with generous safe margins. "
    "Isolated subject on a perfectly flat pure chroma-key magenta background (#FF00FF). "
    "No white outline: a consistent white sticker outline will be added during post-processing. "
    "Absolutely no magenta objects, fur, clothing, props or reflections on the character. "
    "No words, letters, typography, watermark, signature, logo, collage, grid, multiple views or scenery."
)
NEGATIVE_PROMPT = (
    "text, letters, words, typography, watermark, signature, logo, multiple characters, duplicate character, "
    "collage, contact sheet, grid, split panel, cropped body, cropped ears, cropped hands, extra limbs, extra fingers, "
    "malformed paws, distorted eyes, different eye colors, different character design, realistic human, scary, angry, horror, "
    "cluttered background, scenery, gradient background, patterned background, magenta fur, magenta clothes, magenta props, "
    "low resolution, blur, noise, jpeg artifacts, oversharpening"
)

REFERENCE_PROMPT = CHARACTER_LOCK + ". " + (
    "Neutral character reference image: Omnii is standing upright, facing the viewer, full body visible, both tiny paws relaxed, "
    "gentle closed-mouth smile, no props, no floating symbols, symmetrical easy-to-recognize pose. "
) + COMPOSITION_LOCK

def generate_image(prompt, seed, reference=None, steps=25, guidance=7.0, adapter_scale=0.65):
    generator = torch.Generator(device="cpu").manual_seed(int(seed))
    kwargs = dict(
        prompt=prompt,
        negative_prompt=NEGATIVE_PROMPT,
        num_inference_steps=steps,
        guidance_scale=guidance,
        width=512,
        height=512,
        generator=generator,
    )
    if reference is not None:
        kwargs["ip_adapter_image"] = reference
        if hasattr(pipe, "set_ip_adapter_scale"):
            pipe.set_ip_adapter_scale(adapter_scale)
    return pipe(**kwargs).images[0]


In [ ]:
# STAGE 1 — make four neutral Omnii reference candidates.
# Review all four. You choose the best one manually in the next cell.
reference_files = sorted(REF_DIR.glob("reference_*.png"))
if len(reference_files) < 4:
    for idx in range(len(reference_files), 4):
        seed = BASE_SEED + idx
        image = generate_image(REFERENCE_PROMPT, seed=seed, reference=None, steps=28, guidance=7.0)
        target = REF_DIR / f"reference_{idx+1:02d}.png"
        image.save(target)
        print("Saved:", target.name)
        display(image)
else:
    print("Found existing reference candidates; displaying them:")
    for target in reference_files:
        print(target.name)
        display(Image.open(target))


In [ ]:
# STAGE 1B — choose the reference you like best (0 = first, 1 = second, and so on).
# Change REFERENCE_INDEX after reviewing the four candidates, then run this cell.
REFERENCE_INDEX = 0

candidate_paths = sorted(REF_DIR.glob("reference_*.png"))
if len(candidate_paths) < 4:
    raise RuntimeError("Run the previous cell to create the reference candidates first.")
if REFERENCE_INDEX not in range(4):
    raise ValueError("REFERENCE_INDEX must be 0, 1, 2, or 3.")

REFERENCE_PATH = candidate_paths[REFERENCE_INDEX]
reference_image = Image.open(REFERENCE_PATH).convert("RGB")
print("Selected reference:", REFERENCE_PATH.name)
display(reference_image)


In [ ]:
# STAGE 2 — load IP-Adapter. It conditions later generations on the chosen Omnii reference.
IP_ADAPTER_READY = False
try:
    pipe.load_ip_adapter(ADAPTER_REPO, subfolder="models", weight_name=ADAPTER_FILE)
    pipe.set_ip_adapter_scale(0.65)
    IP_ADAPTER_READY = True
    print("IP-Adapter loaded. Reference strength = 0.65")
except Exception as exc:
    print("IP-Adapter did not load:", repr(exc))
    print("Do not generate the full pack yet. Fix the dependency/model access issue and rerun this cell.")
    print("Reference conditioning is strongly recommended for a consistent character.")


## Stage 3 — контрольная генерация

Сначала проверь только три разных действия: приветствие, смех и задумчивость. Если персонаж заметно меняется, не запускай весь пак. Попробуй поменять силу IP-Adapter с 0.65 на 0.75 или 0.55, затем перегенерируй тесты.

In [ ]:
TEST_SCENES = [
    ("01-hello", "Omnii happily greets a friend, standing upright and raising one tiny paw in a clear friendly wave, bright welcoming eyes and a small warm smile."),
    ("06-laugh", "Omnii laughs joyfully with eyes squeezed into happy arcs, mouth open in a delighted laugh, tiny tears of laughter at the outer corners, body slightly bouncing."),
    ("09-thinking", "Omnii sits slightly turned, one tiny paw touching its chin, eyes looking upward to one side, thoughtful curious expression.")
]

if not IP_ADAPTER_READY:
    raise RuntimeError("IP-Adapter is not loaded. Resolve the error above before continuing.")

for index, (name, action) in enumerate(TEST_SCENES):
    prompt = CHARACTER_LOCK + ". " + action + " " + COMPOSITION_LOCK
    image = generate_image(prompt, seed=BASE_SEED + 100 + index, reference=reference_image, steps=28, guidance=7.0, adapter_scale=0.65)
    target = TEST_DIR / f"{name}.png"
    image.save(target)
    print("Saved:", target.name)
    display(image)


## Stage 4 — полный набор из 24 стикеров

Сначала оцени три теста. По умолчанию полный рендер выключен. Поменяй **RUN_FULL_PACK** на **True** вручную только после того, как устраивает стабильность персонажа. Не закрывай сессию до скачивания результата.

In [ ]:
RUN_FULL_PACK = False  # Switch to True only after you approve the three test images.
VARIANTS_PER_STICKER = 1  # Increase to 2 only if you have enough GPU quota.

STICKER_PROMPTS = [
    ("01-hello", "Omnii happily greets a friend and raises one tiny paw in a clear friendly wave. Bright welcoming eyes, warm smile."),
    ("02-happy", "Omnii is bursting with joy, eyes smiling, delighted open grin, both paws lifted close to its cheeks, a few tiny golden sparkles."),
    ("03-thanks", "Omnii carefully holds a small soft-gold heart between both paws, looking grateful and tender, slight head tilt."),
    ("04-love", "Omnii hugs one large soft red heart against its fluffy chest, affectionate shining eyes, two or three tiny hearts nearby."),
    ("05-hug", "Omnii opens both tiny arms wide and leans forward slightly to invite a hug, kind bright expression."),
    ("06-laugh", "Omnii laughs joyfully with happy curved eyes, open grin and tiny tears of laughter, body slightly bouncing."),
    ("07-cry", "Omnii looks tenderly sad with one large clear tear rolling down a cheek, body lowered and paws held close."),
    ("08-sorry", "Omnii bows its fluffy head slightly, brings both paws together apologetically, and looks upward with gentle pleading eyes."),
    ("09-thinking", "Omnii sits slightly turned with one paw touching its chin, eyes looking upward to one side, thoughtful and curious."),
    ("10-idea", "Omnii suddenly has a bright idea, raises one paw while a small elegant soft-gold light bulb floats above it, excited eyes."),
    ("11-wow", "Omnii is amazed, eyes very wide and mouth forming a small round expression, paws lifted slightly, a few blue-white sparkles."),
    ("12-yes", "Omnii gives a clear enthusiastic thumbs-up with one tiny paw, standing confidently and smiling."),
    ("13-no", "Omnii gently crosses both tiny paws in front of its chest to signal no, with a firm but kind expression."),
    ("14-shrug", "Omnii raises both paws slightly and gives a small shoulder shrug, head tilted, eyebrows lifted in a puzzled expression."),
    ("15-support", "Omnii offers a small pale-blue glowing heart forward with both paws, caring and reassuring expression."),
    ("16-coffee", "Omnii holds a tiny plain warm ceramic cup with both paws, enjoying a cozy break, small curl of steam above the cup."),
    ("17-sleep", "Omnii curls into a soft round sleeping pose on a tiny pale-blue crescent pillow, eyes peacefully closed, one small star."),
    ("18-morning", "Omnii stretches both tiny arms after waking, smiling sleepily, a small soft-gold sun floating nearby."),
    ("19-party", "Omnii celebrates with both paws raised and a wide smile, a few blue, emerald and soft-gold confetti pieces."),
    ("20-wait", "Omnii sits patiently with paws resting on its knees, calm expectant expression, a tiny simple hourglass beside it."),
    ("21-work", "Omnii sits behind a tiny plain open laptop, focused but cheerful, one paw raised as if saying it is working on it."),
    ("22-done", "Omnii stands proudly with a satisfied smile and a small confident nod, a clean emerald check mark floating beside its head."),
    ("23-peace", "Omnii raises one paw in a clear two-finger peace sign, tilts its head playfully and smiles."),
    ("24-bye", "Omnii turns slightly to one side and waves goodbye with one paw, warm smile, tiny blue-gold sparkles.")
]

if not RUN_FULL_PACK:
    print("Full pack is OFF. Change RUN_FULL_PACK = True after reviewing the test images.")
elif not IP_ADAPTER_READY:
    raise RuntimeError("IP-Adapter must be loaded before bulk generation.")
else:
    total = len(STICKER_PROMPTS) * VARIANTS_PER_STICKER
    done = 0
    for item_index, (name, action) in enumerate(STICKER_PROMPTS):
        for variant in range(1, VARIANTS_PER_STICKER + 1):
            filename = f"{name}-v{variant}.png"
            target = SOURCE_DIR / filename
            if target.exists():
                print("Skipping existing", filename)
                done += 1
                continue
            prompt = CHARACTER_LOCK + ". " + action + " " + COMPOSITION_LOCK
            seed = BASE_SEED + 1000 + item_index * 10 + variant
            image = generate_image(prompt, seed=seed, reference=reference_image, steps=28, guidance=7.0, adapter_scale=0.65)
            image.save(target)
            done += 1
            print(f"[{done}/{total}] Saved {filename}")
            display(image)
    print("Generation pass finished. Review every candidate before publication.")


## Stage 5 — подготовить прозрачные стикеры и ZIP

Эта ячейка запускается после полного рендера. Она использует RemBG для удаления фона, добавляет единый белый контур и мягкую тень, сохраняет PNG/WebP 512×512 и собирает ZIP. Автоматическое удаление фона не идеально: обязательно проверь края и все лапки.

In [ ]:
from rembg import remove, new_session
import numpy as np

source_images = sorted(SOURCE_DIR.glob("*.png"))
if len(source_images) < 24:
    raise RuntimeError(f"В папке source сейчас {len(source_images)} PNG. Нужно минимум 24 изображения. Выполни полный рендер и проверь файлы.")

print("Starting background removal for", len(source_images), "images. First image may take longer while the model downloads.")
rmbg_session = new_session("isnet-general-use")

def make_sticker(source_path, output_path):
    original = Image.open(source_path).convert("RGBA")
    cutout = remove(original, session=rmbg_session).convert("RGBA")
    alpha = cutout.getchannel("A")
    bbox = alpha.getbbox()
    if not bbox:
        raise ValueError(f"No foreground detected in {source_path.name}")
    cutout = cutout.crop(bbox)
    cutout.thumbnail((472, 472), Image.Resampling.LANCZOS)
    foreground = Image.new("RGBA", (512, 512), (0, 0, 0, 0))
    x = (512 - cutout.width) // 2
    y = (512 - cutout.height) // 2
    foreground.alpha_composite(cutout, (x, y))
    mask = foreground.getchannel("A")
    expanded = mask.filter(ImageFilter.MaxFilter(15))
    outline = Image.new("RGBA", (512, 512), (255, 255, 255, 0))
    outline.putalpha(expanded)
    shadow_alpha = expanded.filter(ImageFilter.GaussianBlur(4))
    shadow_alpha = ImageChops.offset(shadow_alpha, 0, 3).point(lambda value: int(value * 0.22))
    shadow = Image.new("RGBA", (512, 512), (18, 44, 70, 0))
    shadow.putalpha(shadow_alpha)
    result = Image.new("RGBA", (512, 512), (0, 0, 0, 0))
    result.alpha_composite(shadow)
    result.alpha_composite(outline)
    result.alpha_composite(foreground)
    result.save(output_path, format="PNG", optimize=True)
    return result

for source in source_images:
    target_png = FINAL_DIR / (source.stem.replace("-v1", "") + ".png")
    try:
        ready = make_sticker(source, target_png)
        target_webp = target_png.with_suffix(".webp")
        ready.save(target_webp, format="WEBP", lossless=True, method=6)
        if target_webp.stat().st_size > 512 * 1024:
            ready.save(target_webp, format="WEBP", lossless=False, quality=92, method=6)
        print(f"{source.name} -> {target_webp.name}: {target_webp.stat().st_size / 1024:.1f} KB")
    except Exception as exc:
        print("REVIEW REQUIRED:", source.name, repr(exc))


In [ ]:
# Create a contact sheet and a manifest. Review all thumbnails before any publishing.
webp_files = sorted(FINAL_DIR.glob("*.webp"))
if not webp_files:
    raise RuntimeError("No WebP files found. Run the transparency/export cell first.")

thumb = 160
columns = 4
rows = math.ceil(len(webp_files) / columns)
sheet = Image.new("RGB", (columns * 220, rows * 205), (240, 245, 250))
draw = ImageDraw.Draw(sheet)
for idx, path in enumerate(webp_files):
    img = Image.open(path).convert("RGBA")
    img.thumbnail((thumb, thumb), Image.Resampling.LANCZOS)
    x = (idx % columns) * 220 + 30
    y = (idx // columns) * 205 + 5
    checker = Image.new("RGBA", (thumb, thumb), (250, 250, 250, 255))
    checker.alpha_composite(img, ((thumb - img.width)//2, (thumb - img.height)//2))
    sheet.paste(checker.convert("RGB"), (x, y))
    draw.text((x, y + thumb + 5), path.name[:27], fill=(24, 51, 77))
preview_path = WORK / "preview.png"
sheet.save(preview_path)
display(sheet)

manifest = {
    "schema_version": "1.0",
    "pack": {
        "id": "qun_omnii_01",
        "title": "Omnii — добро рядом",
        "version": "0.1-candidate",
        "artist": "QUN Studio (AI-assisted; review and credit to be confirmed)",
        "model": MODEL_ID,
        "adapter": ADAPTER_REPO + "/" + ADAPTER_FILE,
        "notes": "AI-generated candidate pack; not published. Human review, rights review and edits are pending."
    },
    "stickers": []
}
for path in webp_files:
    with Image.open(path) as img:
        manifest["stickers"].append({
            "file": path.name,
            "format": "webp",
            "width": img.width,
            "height": img.height,
            "bytes": path.stat().st_size,
            "emoji": "🙂",
            "status": "HUMAN REVIEW"
        })
manifest_path = WORK / "manifest.json"
manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
print("Preview:", preview_path)
print("Manifest:", manifest_path)
print("Files:", len(webp_files))


In [ ]:
# Package transparent exports, originals, preview, and manifest for download.
zip_path = WORK / "qun_omnii_01_candidate_pack.zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=5) as archive:
    for folder_name, folder in [("source", SOURCE_DIR), ("telegram", FINAL_DIR)]:
        for path in sorted(folder.iterdir()):
            if path.is_file():
                archive.write(path, arcname=f"{folder_name}/{path.name}")
    for path in [preview_path, manifest_path]:
        if path.exists():
            archive.write(path, arcname=path.name)
    archive.writestr(
        "QA_TO_DO.txt",
        "Before publication: inspect all stickers for character consistency and background-removal artifacts; "
        "verify file size <= 512 KB; assign accurate emoji; review model and adapter licenses; confirm rights and authorship. "
        "This pack is a candidate export and has not been published."
    )

print("ZIP:", zip_path)
print("Size (MB):", round(zip_path.stat().st_size / (1024*1024), 2))
try:
    from IPython.display import FileLink
    display(FileLink(str(zip_path)))
except Exception:
    print("Download the ZIP from the Kaggle Output / working files panel.")


## После генерации

1. Просмотри preview.png целиком и открой отдельные стикеры на светлом и тёмном фоне.
2. Если RemBG удалил части белого меха или обводки, доработай эти изображения вручную.
3. Проверь размер каждого WebP: не более 512 КБ, холст 512×512 px, фон прозрачен.
4. Скачай qun_omnii_01_candidate_pack.zip до выключения Kaggle-сессии.
5. Только после человеческой проверки можно подготовить финальный пак для QUN и отдельно опубликовать его в Telegram через @Stickers.

**Важно:** первая серия считается кандидатом, а не утверждённым паком. Манифест намеренно помечает изображения как HUMAN REVIEW.